# CS 691 · Module 05 · From a photograph to a symbolic claim

Explain what a learned score measures and decide which claim the desk can record from it.

**Before you begin.** You can write a fact about a named object and trace a conclusion to its evidence. We now ask where a fact might come from when the evidence is a photograph.

Ava’s receipt identifies C1 as returned at 9. It says nothing about its accessories. If you want to know whether the cap was visible at return, which evidence would you seek?

<details><summary>Reveal the answer</summary>

Seek an inspection made at return or a photograph of C1 from that return. The time and camera on the receipt help link the new evidence to the right event.

</details>

## What can the desk see?

Ava returns camera C1. The lending desk has a record of the return and a photograph of the camera. The record identifies the item. To tell whether the lens cap is visible, someone must interpret the image. A perception model is a program that learns to recognize patterns in examples.

Returned(Ava, C1) and CapVisible(C1) answer different questions. A return receipt supplies evidence for the first. The desk still needs evidence about the cap.

**Predict before running the cell.** What does the second False tell us about the cap?

In [ ]:
receipt = {("Returned", "Ava", "C1")}
return_fact = ("Returned", "Ava", "C1")
cap_fact = ("CapVisible", "C1")
print(return_fact in receipt, cap_fact in receipt)

<details><summary>Reveal output and explanation</summary>

```text
True False
```

The receipt contains no such statement. The cap might be present, missing, or outside the photograph. We need another source to distinguish those possibilities.

</details>

## How two visual cues become a score

A feature is a number describing an input. For this small example, record 1 when a cue is visible and 0 otherwise. A neural unit multiplies each feature by a weight, adds a starting offset called a bias, and applies a function. Our function, ReLU, keeps a positive total and replaces a negative total with zero. We will learn how a weight is fitted in Module 07.

Use two cues, a circular rim and a dark center. With both visible, the example unit gives 0.6 × 1 + 0.3 × 1 − 0.1 = 0.8. The weights here are supplied for the calculation. A high score means that the cues resemble those associated with caps in this example.

**Predict before running the cell.** If glare hides the dark center, what score will the same unit give?

In [ ]:
rim, dark_center = 1, 1
weighted_total = 0.6 * rim + 0.3 * dark_center - 0.1
score = max(0.0, weighted_total)
print(f"{score:.2f}")

<details><summary>Reveal output and explanation</summary>

```text
0.80
```

Keep the rim at 1 and change the center to 0. The total is 0.6 + 0 − 0.1 = 0.5, so ReLU returns 0.5.

</details>

## Decide which claim crosses the boundary

A threshold is a cutoff chosen for an action. In our desk example, a score of at least 0.75 creates CapCandidate for that item. The name tells the next rule that a detector proposed the claim. A staff inspection would give a separate fact, CapConfirmed.

C1 scores 0.80 and becomes a candidate. A score of 0.74 sends an item to inspection. The desk can still learn that the cap is present on either item.

**Predict before running the cell.** Would raising the cutoff to 0.81 establish that C1 has lost its cap?

In [ ]:
scores = {"C1": 0.80, "C2": 0.74, "C3": 0.75}
for item, score in scores.items():
    action = "CapCandidate" if score >= 0.75 else "Inspect"
    print(item, action)

<details><summary>Reveal output and explanation</summary>

```text
C1 CapCandidate
C2 Inspect
C3 CapCandidate
```

No. C1's photo and score stay the same. The higher cutoff changes the desk's action to inspection.

</details>

## Check the cutoff against inspected photographs

A label is a reference answer for a particular example. Staff inspected four returns and recorded whether a cap was visible in each photograph. Compare the proposed candidates with these labels. Every incorrect proposal or missed cap tells us something the score alone cannot tell us.

At 0.75, the desk proposes C1 and C2. Staff see a cap on C1 and C3. C2 is a false positive, a proposed cap absent from its reference label. C3 is a false negative, a labeled cap the detector missed.

**Predict before running the cell.** What changes on these four photographs if the cutoff falls from 0.75 to 0.70?

In [ ]:
# Each row holds item, score and staff label.
rows = [("C1", .80, True), ("C2", .78, False),
        ("C3", .70, True), ("C4", .30, False)]
for item, score, label in rows:
    proposed = score >= .75
    if proposed != label:
        print(item, "false positive" if proposed else "false negative")

<details><summary>Reveal output and explanation</summary>

```text
C2 false positive
C3 false negative
```

C3 now becomes a candidate, so its miss disappears. C2 still crosses the cutoff and is still a false positive. This small check describes these four cases.

</details>

## Build one photo score from two visible cues

A one marks a visible cue and a zero marks an absent cue in the input. Multiplying each input by its weight gives its contribution to the score. The code adds those contributions, subtracts the offset, applies a floor of zero and prints two decimal places.

Replace this line:
rim, dark_center = 1, 1
with:
rim, dark_center = 0, 0
What will print?

In [ ]:
rim, dark_center = 1, 1
weighted_total = 0.6 * rim + 0.3 * dark_center - 0.1
score = max(0.0, weighted_total)
print(f"{score:.2f}")

<details><summary>Worked answer after your change</summary>

0.00

Both weighted contributions become zero. The total of −0.1 lies below the floor, so the displayed score is zero.

</details>

## Exercise 1

A delivery note records that Ava returned C2. Write one fact justified by the note and one question it leaves open.

<details><summary>Hint</summary>

Separate the item identifier from its physical condition.

</details>

<details><summary>Worked answer</summary>

Returned(Ava, C2) is justified. Whether C2 has a visible lens cap remains open until someone checks the item or its photograph.

</details>

## Exercise 2

With the two-cue unit, find the score when the rim is absent and the dark center is visible. What action follows at 0.75?

<details><summary>Hint</summary>

Use rim = 0 and center = 1 before applying ReLU.

</details>

<details><summary>Worked answer</summary>

The total is 0 + 0.3 − 0.1 = 0.2. ReLU keeps 0.2. The item goes to inspection.

</details>

## Exercise 3

A detector scores three photos 0.9, 0.7 and 0.6. Their labels are cap present, cap absent and cap present. Compare cutoffs of 0.7 and 0.8 inclusive.

<details><summary>Hint</summary>

For each cutoff, first list the proposed photos. Then compare that list with the first and third labels.

</details>

<details><summary>Worked answer</summary>

At 0.7 the first two are candidates, so the second is a false positive and the third a false negative. At 0.8 only the first is a candidate. The false positive disappears, while the third photo is still missed.

</details>

C1 now has a proposed cap fact. How can a rule combine it with Ava’s reservation without accidentally using another camera’s photograph?

Two photos score above the cutoff, one for C1 and one for C3. Ava reserved C3. Which information must the detector pass along with each candidate so the desk can choose the matching evidence?

<details><summary>Reveal the answer</summary>

Keep the camera ID with each candidate. Matching C3 in the reservation to C3 in the candidate identifies the relevant photograph even if both scores are identical.

</details>